In [ ]:
import os, re, json, random, numpy as np, pandas as pd, torch, emoji
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

import evaluate
from datasets import Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer,
                          EarlyStoppingCallback)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cuda'

In [ ]:
TRAIN_XLSX = "/content/drive/MyDrive/Emotion/train_nor_811.xlsx"
VAL_XLSX   = "/content/drive/MyDrive/Emotion/valid_nor_811.xlsx"
TEST_XLSX  = "/content/drive/MyDrive/Emotion/test_nor_811.xlsx"

train_df = pd.read_excel(TRAIN_XLSX)
val_df   = pd.read_excel(VAL_XLSX)
test_df  = pd.read_excel(TEST_XLSX)

for name,df in [("train",train_df),("valid",val_df),("test",test_df)]:
    assert len(df)>0, f"{name} rỗng!"
    assert any(c.lower()=="sentence" for c in df.columns), f"{name} thiếu cột Sentence"
    assert any(c.lower()=="emotion"  for c in df.columns), f"{name} thiếu cột Emotion"


In [ ]:
def clean_text(s: str) -> str:
    if not isinstance(s, str): s = str(s)
    s = s.replace("\u200d"," ").replace("…","...")
    s = re.sub(r"\s+", " ", s).strip().lower()
    # giữ tín hiệu cảm xúc: chuyển emoji -> token chữ
    s = emoji.demojize(s, language="en")
    return s

def normalize(df):
    df = df.rename(columns={c:c.strip().title() for c in df.columns})
    df = df[["Sentence","Emotion"]].copy()
    df["Sentence"] = df["Sentence"].astype(str).map(clean_text)
    df["Emotion"]  = df["Emotion"].astype(str).str.strip()
    return df.dropna()

train_df = normalize(train_df)
val_df   = normalize(val_df)
test_df  = normalize(test_df)

print(train_df.head(3))


                                            Sentence  Emotion
0              cho mình xin bài nhạc tên là gì với ạ    Other
1  cho đáng đời con quỷ . về nhà lôi con nhà mày ...  Disgust
2  lo học đi . yêu đương lol gì hay lại thích học...  Disgust


In [ ]:
# gom nhãn từ cả 3 để encoder nhất quán
all_labels = pd.concat([train_df["Emotion"], val_df["Emotion"], test_df["Emotion"]], axis=0)
le = LabelEncoder().fit(all_labels)

for df in (train_df, val_df, test_df):
    df["label"] = le.transform(df["Emotion"])

num_labels = len(le.classes_)
label2id = {c:i for i,c in enumerate(le.classes_)}
id2label = {i:c for c,i in label2id.items()}
num_labels, label2id


(7,
 {'Anger': 0,
  'Disgust': 1,
  'Enjoyment': 2,
  'Fear': 3,
  'Other': 4,
  'Sadness': 5,
  'Surprise': 6})

In [ ]:
from datasets import DatasetDict, Dataset
MODEL_NAME = "vinai/phobert-base-v2"

datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df[["Sentence","label"]].rename(columns={"Sentence":"text"}), preserve_index=False),
    "validation": Dataset.from_pandas(val_df[["Sentence","label"]].rename(columns={"Sentence":"text"}), preserve_index=False),
    "test": Dataset.from_pandas(test_df[["Sentence","label"]].rename(columns={"Sentence":"text"}), preserve_index=False),
})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
MAX_LEN = 128

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

tokenized = datasets.map(tokenize_fn, batched=True, remove_columns=["text"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/5548 [00:00<?, ? examples/s]

Map:   0%|          | 0/686 [00:00<?, ? examples/s]

Map:   0%|          | 0/693 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5548
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 686
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 693
    })
})

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_labels),
    y=train_df["label"].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float, device=device)
class_weights


tensor([2.0270, 0.7400, 0.5087, 2.4924, 0.7763, 0.8369, 3.2751],
       device='cuda:0')

In [ ]:
!pip install evaluate --upgrade

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id
).to(device)

metric_acc = evaluate.load("accuracy")
metric_f1  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":   metric_acc.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro":   metric_f1.compute(predictions=preds, references=labels, average="macro")["f1"],
        "f1_micro":   metric_f1.compute(predictions=preds, references=labels, average="micro")["f1"],
    }

OUTPUT_DIR = "/content/drive/MyDrive/Emotion/output"
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    num_train_epochs=10,                   # early stopping sẽ cắt sớm nếu đủ
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

from torch.nn import CrossEntropyLoss
from transformers import Trainer
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels),
                        labels.view(-1))
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
train_result = trainer.train()
train_result


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: doraemon207078 (doraemon207078-h-c-vi-n-c-ng-ngh-b-u-ch-nh-vi-n-th-ng) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Micro
200,1.867100,1.798774,0.424198,0.296602,0.424198
400,1.427300,1.395437,0.453353,0.440224,0.453353
600,1.229300,1.208803,0.574344,0.530704,0.574344
800,0.891300,1.086065,0.618076,0.593028,0.618076
1000,0.926500,1.100706,0.578717,0.548332,0.578717
1200,0.610600,1.189240,0.599125,0.566413,0.599125
1400,0.594600,1.129984,0.648688,0.590371,0.648688
1600,0.428500,1.116135,0.642857,0.612068,0.642857
1800,0.343100,1.318100,0.645773,0.604698,0.645773
2000,0.343800,1.320319,0.650146,0.606859,0.650146


TrainOutput(global_step=3400, training_loss=0.6215169944482691, metrics={'train_runtime': 642.4083, 'train_samples_per_second': 86.363, 'train_steps_per_second': 5.402, 'total_flos': 1506964274229000.0, 'train_loss': 0.6215169944482691, 'epoch': 9.798270893371757})

In [ ]:
metrics = trainer.evaluate(tokenized["test"])
print(metrics)

preds = trainer.predict(tokenized["test"]).predictions.argmax(-1)
y_true = test_df["label"].to_numpy()
print(classification_report(y_true, preds, target_names=le.classes_, digits=4))


{'eval_loss': 1.218714714050293, 'eval_accuracy': 0.6825396825396826, 'eval_f1_macro': 0.6647320453754212, 'eval_f1_micro': 0.6825396825396826, 'eval_runtime': 0.4953, 'eval_samples_per_second': 1399.046, 'eval_steps_per_second': 44.414, 'epoch': 9.798270893371757}
              precision    recall  f1-score   support

       Anger     0.5476    0.5750    0.5610        40
     Disgust     0.6328    0.6136    0.6231       132
   Enjoyment     0.7565    0.7565    0.7565       193
        Fear     0.6604    0.7609    0.7071        46
       Other     0.5971    0.6434    0.6194       129
     Sadness     0.8587    0.6810    0.7596       116
    Surprise     0.5652    0.7027    0.6265        37

    accuracy                         0.6825       693
   macro avg     0.6598    0.6762    0.6647       693
weighted avg     0.6917    0.6825    0.6846       693



In [ ]:
BEST_DIR = os.path.join(OUTPUT_DIR, "best")
os.makedirs(BEST_DIR, exist_ok=True)
trainer.save_model(BEST_DIR)
tokenizer.save_pretrained(BEST_DIR)

with open(os.path.join(BEST_DIR, "id2label.json"), "w", encoding="utf-8") as f:
    json.dump(id2label, f, ensure_ascii=False, indent=2)
with open(os.path.join(BEST_DIR, "label2id.json"), "w", encoding="utf-8") as f:
    json.dump(label2id, f, ensure_ascii=False, indent=2)

print("Saved to:", BEST_DIR)


Saved to: /content/drive/MyDrive/Emotion/output/best


In [ ]:
import torch.onnx as onnx_export
from pathlib import Path

onnx_path = Path(BEST_DIR) / "model.onnx"
model_onnx = AutoModelForSequenceClassification.from_pretrained(BEST_DIR).to("cpu").eval()

# dummy input
dummy = tokenizer("demo text", return_tensors="pt", truncation=True, max_length=MAX_LEN)
input_names  = ["input_ids", "attention_mask"]
output_names = ["logits"]
dynamic_axes = {
    "input_ids":      {0: "batch", 1: "sequence"},
    "attention_mask": {0: "batch", 1: "sequence"},
    "logits":         {0: "batch"}
}

onnx_export.export(
    model_onnx,                # cpu để export ổn định
    (dummy["input_ids"], dummy["attention_mask"]),
    f"{onnx_path}",
    input_names=input_names,
    output_names=output_names,
    dynamic_axes=dynamic_axes,
    opset_version=14,
    do_constant_folding=True
)
print("ONNX saved to:", onnx_path)


/tmp/ipython-input-3237006468.py:17: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  onnx_export.export(


ONNX saved to: /content/drive/MyDrive/Emotion/output/best/model.onnx


In [ ]:
import onnxruntime as ort
import numpy as np
import torch.nn.functional as F

# tiện ích suy luận với PyTorch (best)
def torch_predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    with torch.no_grad():
        logits = model_onnx(**inputs).logits
    probs = F.softmax(logits, dim=-1).numpy()[0]
    return probs

# tiện ích suy luận với ONNX
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])

def onnx_predict(text):
    inputs = tokenizer(text, return_tensors="np", truncation=True, max_length=MAX_LEN)
    ort_inputs = {
        "input_ids":      inputs["input_ids"].astype("int64"),
        "attention_mask": inputs["attention_mask"].astype("int64"),
    }
    logits = session.run(["logits"], ort_inputs)[0]
    probs = (np.exp(logits) / np.exp(logits).sum(-1, keepdims=True))[0]
    return probs

sample = "Mày ngu mày chết mẹ mày đi"
pt, ox = torch_predict(sample), onnx_predict(sample)
print("PT top:", id2label[int(pt.argmax())], float(pt.max()))
print("ONNX top:", id2label[int(ox.argmax())], float(ox.max()))
print("L2 diff:", float(np.linalg.norm(pt-ox)))


PT top: Anger 0.9730133414268494
ONNX top: Anger 0.9730134606361389
L2 diff: 1.202071473471733e-07
